# Лабораторная работа 1
## Методы агрегации и группировки данных
Вариант 2. Набор данных drivers.csv.

Выполнил: Воронов Владислав Владимирович. Группа: 4311з.

Дисциплина: Введение в анализ данных. Кафедра прикладной информатики (№ 41).

Преподаватель: Боженко Виктория Вячеславовна.

Цель работы: овладение навыками агрегации и группировки табличных данных для преобразования детальных записей в структурированные аналитические отчёты.

Выполнены предварительная обработка, шесть заданий варианта 2 и собственная группировка с фильтрацией. Для каждой операции приведены код, результат и интерпретация. Дополнительные задания после защиты не входят в эту работу.

Результаты, использующие MILES, рассчитаны при явно указанной гипотезе восстановления повреждённых чисел. Подтверждённого исходного столбца расстояний нет. До сверки с ним выводы о расстояниях следует считать условными.

Ссылка на репозиторий с ноутбуком: [добавить после загрузки на GitHub и проверки доступа преподавателя].

## 1 Исходные данные и загрузка
CSV содержит дату и время начала START_DATE, дату и время окончания END_DATE, категорию CATEGORY*, место начала START, место окончания STOP, расстояние в милях MILES и цель поездки PURPOSEroute. Одна строка описывает поездку. Используются разделитель «;» и кодировка UTF-8 с BOM.

Исходный файл не изменяется. Подготовленный набор и результаты сохраняются отдельно. Для воспроизведения нужно поместить drivers.csv рядом с ноутбуком и выполнить ячейки сверху вниз.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
raw = pd.read_csv('drivers.csv', sep=';', encoding='utf-8-sig')
print(f'Python-совместимый анализ; pandas {pd.__version__}')
print(f'Размер исходной таблицы: {raw.shape[0]} строк, {raw.shape[1]} столбцов')
print(raw.head(3).T.to_string())
print('\nТипы данных:')
print(raw.dtypes.to_string())

Python-совместимый анализ; pandas 2.2.3
Размер исходной таблицы: 161 строк, 7 столбцов
                             0                 1                 2
START_DATE    01.10.2016 19:12  01.11.2016 13:32  01.12.2016 12:33
END_DATE      01.10.2016 19:32  01.11.2016 13:46  01.12.2016 12:49
CATEGORY*             Business          Business          Business
START                  Midtown           Midtown           Midtown
STOP               East Harlem      Midtown East     Hudson Square
MILES                  44963.0           45108.0           45170.0
PURPOSEroute           MEETING    Meal/Entertain    Meal/Entertain

Типы данных:
START_DATE       object
END_DATE         object
CATEGORY*        object
START            object
STOP             object
MILES           float64
PURPOSEroute     object


## 2 Предварительная обработка
### 2.1 Пропуски и дубликаты
Проверяются пропуски и полные повторы строк. Имена столбцов приводятся к единому виду: CATEGORY* заменяется на CATEGORY, PURPOSEroute на PURPOSE. В текстовых полях удаляются пробелы по краям, в категории и цели нормализуется регистр. Названия географических точек не объединяются по предположению.

Повторяющиеся строки с одинаковыми значениями всех семи полей считаются техническими дублями. Неизвестную цель поездки нельзя достоверно восстановить по её категории. Поэтому пропуски обозначаются отдельным значением «Не указана». Такая группа сохраняет поездки при подсчёте количества, но не обозначает реальную цель.

In [2]:
print('Пропуски до обработки:')
print(raw.isna().sum().to_string())
print('Полные дубли сверх первого экземпляра:', raw.duplicated().sum())
df = raw.rename(columns={'CATEGORY*': 'CATEGORY', 'PURPOSEroute': 'PURPOSE'}).copy()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().replace('', pd.NA)
df['CATEGORY'] = df['CATEGORY'].str.title()
df['PURPOSE'] = df['PURPOSE'].str.title()
df = df.drop_duplicates().reset_index(drop=True)
df['PURPOSE_WAS_MISSING'] = df['PURPOSE'].isna()
df['PURPOSE'] = df['PURPOSE'].fillna('Не указана')
print('\nКатегории после удаления дублей:')
print(df['CATEGORY'].value_counts().to_string())
print('Осталось строк:', len(df))
print('Цель не указана:', df['PURPOSE_WAS_MISSING'].sum())

Пропуски до обработки:
START_DATE       0
END_DATE         0
CATEGORY*        0
START            0
STOP             0
MILES            0
PURPOSEroute    77
Полные дубли сверх первого экземпляра: 2

Категории после удаления дублей:
CATEGORY
Business    148
Personal     11
Осталось строк: 159
Цель не указана: 77


В исходном наборе 161 строка и 7 столбцов. Удалены 2 полных дубля; осталось 159 поездок: 148 деловых и 11 личных. Сохранились 77 поездок с неизвестной целью, то есть 48,43% очищенного набора. Значения BUSINESS и Business объединены; MEETING и Meeting также приведены к одному написанию.

### 2.2 Даты и длительность
Даты имеют формат месяц.день.год: например, 1.13.2016 означает 13 января. Один формат применяется ко всему столбцу, включая неоднозначные записи 01.10.2016. Длительность вычисляется как разность конца и начала в минутах. Полная дата позволяет корректно обработать поездку через полночь.

Проверяются ошибки преобразования и неположительная длительность. При обнаружении такой ошибки расчёт останавливается, а запись не удаляется незаметно.

In [3]:
for col in ['START_DATE', 'END_DATE']:
    df[col] = pd.to_datetime(df[col], format='%m.%d.%Y %H:%M', errors='coerce')
assert df[['START_DATE', 'END_DATE']].notna().all().all(), 'Ошибка даты'
df['DURATION_MIN'] = (df['END_DATE'] - df['START_DATE']).dt.total_seconds() / 60
assert df['DURATION_MIN'].gt(0).all(), 'Неположительная длительность'
print('Период:', df['START_DATE'].min(), '-', df['START_DATE'].max())
print('Длительность, минуты:')
print(df['DURATION_MIN'].describe().round(2).to_string())
overnight = df['START_DATE'].dt.date != df['END_DATE'].dt.date
print('\nПоездки через полночь:')
print(df.loc[overnight, ['START_DATE', 'END_DATE', 'DURATION_MIN']].to_string(index=False))

Период: 2016-01-10 19:12:00 - 2016-12-14 20:24:00
Длительность, минуты:
count    159.00
mean      20.19
std       21.81
min        1.00
25%        9.50
50%       15.00
75%       22.50
max      206.00

Поездки через полночь:
         START_DATE            END_DATE  DURATION_MIN
2016-07-12 23:47:00 2016-07-13 00:11:00          24.0


### 2.3 Проверка и восстановление расстояний
В исходном файле 135 значений MILES превышают 10 000; после удаления дублей их 133. Эти числа при интерпретации как серийные даты Excel соответствуют 2023 году и месяцам от 1 до 9. Значения 0.8, 0.9, целые расстояния и числа больше 31 при этом сохранились в обычном виде. Это согласуется с гипотезой, что десятичные числа вида 6.2 были автоматически распознаны как день и месяц, а затем выгружены как номера дат.

Принята рабочая гипотеза: серийное число преобразуется в дату с началом отсчёта 1899-12-30, затем расстояние восстанавливается как день + месяц / 10. Например: 44963 → 2023-02-06 → 6.2 мили. Она предполагает одну цифру после десятичного разделителя и порядок день.месяц. Это реконструкция, а не подтверждённый факт об исходных расстояниях.

Столбец MILES_RAW сохраняет исходные числа, MILES_RECOVERED отмечает реконструированные значения. Проверяется обратное преобразование. Оно подтверждает согласованность алгоритма, но не доказывает историческую правильность восстановления. Без подтверждения гипотезы для строгого анализа пришлось бы считать эти 133 расстояния неизвестными.

In [4]:
df['MILES_RAW'] = pd.to_numeric(df['MILES'], errors='coerce')
assert df['MILES_RAW'].notna().all(), 'Нечисловое расстояние'
df['MILES_RECOVERED'] = df['MILES_RAW'].gt(10000)
mask = df['MILES_RECOVERED']
serial = df.loc[mask, 'MILES_RAW']
dates = pd.to_datetime(serial, unit='D', origin='1899-12-30')
assert serial.mod(1).eq(0).all()
assert dates.dt.year.eq(2023).all() and dates.dt.month.between(1, 9).all()
df['MILES'] = df['MILES_RAW'].astype(float)
df.loc[mask, 'MILES'] = dates.dt.day + dates.dt.month / 10
restored = df.loc[mask, 'MILES']
restored_days = np.floor(restored).astype(int)
restored_months = ((restored - restored_days) * 10).round().astype(int)
back_dates = pd.to_datetime(pd.DataFrame({
    'year': 2023, 'month': restored_months, 'day': restored_days}))
reversed_serial = (back_dates - pd.Timestamp('1899-12-30')).dt.days
assert np.array_equal(reversed_serial.to_numpy(), serial.to_numpy())
assert df['MILES'].gt(0).all()
print('Восстановлено:', mask.sum(), 'из', len(df))
print('Без реконструкции:', (~mask).sum())
print(df.loc[mask, ['MILES_RAW', 'MILES']].head(5).to_string(index=False))
print('\nРасстояние при принятой гипотезе, мили:')
print(df['MILES'].describe().round(2).to_string())

Восстановлено: 133 из 159
Без реконструкции: 26
 MILES_RAW  MILES
   44963.0    6.2
   45108.0    1.7
   45170.0    1.9
   45149.0   11.8
   45051.0    5.5

Расстояние при принятой гипотезе, мили:
count    159.00
mean       9.12
std       18.21
min        0.80
25%        3.00
50%        5.50
75%        8.35
max      195.30


### 2.4 Проверка согласованности
Проверяется отношение расстояния к длительности. Порог 100 миль/ч используется только для выявления записей, требующих сверки, и не является правилом удаления. Одна запись даёт 7.6 мили за 2 минуты, то есть 228 миль/ч. Без исходного журнала нельзя определить, какое поле ошибочно, поэтому поездка сохраняется и получает флаг. Интерпретация скоростей в этой работе не выполняется.

Большие расстояния сами по себе не удаляются: поездка на 195.3 мили длится 206 минут, что не противоречит длительной междугородней поездке. Автоматическое удаление по межквартильному размаху исказило бы максимум, который требуется в задании 3.

In [5]:
df['SPEED_MPH'] = df['MILES'] / (df['DURATION_MIN'] / 60)
df['NEEDS_REVIEW'] = df['SPEED_MPH'].gt(100)
print('Записи для сверки:', df['NEEDS_REVIEW'].sum())
print(df.loc[df['NEEDS_REVIEW'],
    ['START_DATE', 'START', 'STOP', 'MILES', 'DURATION_MIN', 'SPEED_MPH']]
    .to_string(index=False))
print('\nПропуски в основных полях после обработки:')
print(df[['START_DATE', 'END_DATE', 'CATEGORY', 'START', 'STOP', 'MILES', 'PURPOSE']]
    .isna().sum().to_string())

Записи для сверки: 1
         START_DATE       START    STOP  MILES  DURATION_MIN  SPEED_MPH
2016-05-18 13:00:00 Morrisville Raleigh    7.6           2.0      228.0

Пропуски в основных полях после обработки:
START_DATE    0
END_DATE      0
CATEGORY      0
START         0
STOP          0
MILES         0
PURPOSE       0


## 3 Задание 1 Количество поездок по категории и точке старта
Нужно сгруппировать поездки по CATEGORY и START и подсчитать число строк в каждой группе. Используется size(), потому что требуется число поездок, независимо от заполненности других признаков.

In [6]:
task1 = df.groupby(['CATEGORY', 'START'], sort=True).size().rename('count')
print(task1.to_string())
print('Всего поездок в группировке:', task1.sum())

CATEGORY  START                
Business  Agnew                     4
          Almond                    1
          Apex                     17
          Arabi                     1
          Arlington                 1
          Briar Meadow              1
          Bryson City               5
          Capitol One               2
          Chapel Hill               2
          College Avenue            1
          Colombo                   8
          Columbia Heights          1
          Galveston                 2
          Georgian Acres            1
          Gulfton                   1
          Hayesville                1
          Lower Garden District     1
          Mandeville                2
          Marigny                   1
          Metairie                  4
          Midtown                  13
          Morrisville              68
          SOMISSPO                  2
          San Jose                  2
          Santa Clara               1
          Savon He

Интерпретация. Наибольшая группа — деловые поездки из Morrisville: 68. Среди личных поездок Morrisville также занимает первое место: 6. Группировка описывает представленную выборку, а не общую популярность точек старта у всех клиентов такси. Сумма групп равна 159, поэтому поездки не потеряны.

## 4 Задание 2 Количество поездок по категории и цели
Поездки группируются по CATEGORY и PURPOSE. Результат преобразуется в DataFrame, столбец количества называется count. Сортировка выполняется по убыванию count, при равенстве — по названиям категории и цели.

In [7]:
task2 = (df.groupby(['CATEGORY', 'PURPOSE']).size()
    .reset_index(name='count')
    .sort_values(['count', 'CATEGORY', 'PURPOSE'], ascending=[False, True, True])
    .reset_index(drop=True))
print(task2.to_string(index=False))

CATEGORY        PURPOSE  count
Business     Не указана     67
Business Meal/Entertain     34
Business Customer Visit     30
Business        Meeting     13
Personal     Не указана     10
Business Temporary Site      4
Personal         Moving      1


Интерпретация. Неизвестная цель образует крупнейшую группу деловых поездок. Среди известных целей деловых поездок наиболее часто встречается Meal/Entertain. Для личных поездок известна цель лишь одной записи — Moving. Поэтому сравнивать структуру целей деловых и личных поездок без учёта пропусков ненадёжно. Точные количества приведены в результате выше.

## 5 Задание 3 Максимальное расстояние по категории
Создаётся сводная таблица pivot_table с функцией max, затем строки сортируются по убыванию MILES. Результат использует реконструированные расстояния.

In [8]:
task3 = (df.pivot_table(index='CATEGORY', values='MILES', aggfunc='max')
    .sort_values('MILES', ascending=False))
print(task3.to_string())

          MILES
CATEGORY       
Business  195.3
Personal   23.8


Интерпретация. Максимум для деловых поездок составляет 195.3 мили, для личных — 23.8 мили. Деловой максимум присутствует в исходном файле как обычное число; личный максимум получен восстановлением. Максимумы характеризуют отдельные крайние наблюдения, а не типичное расстояние поездки.

## 6 Задание 4 Среднее расстояние по категории и цели
В строках сводной таблицы располагается CATEGORY, в столбцах — PURPOSE, значения — среднее MILES. Категории сортируются по убыванию названия. Отсутствующие сочетания сохраняются как NaN: отсутствие поездок не означает нулевое расстояние. Округляется только представление результата.

In [9]:
task4 = (df.pivot_table(index='CATEGORY', columns='PURPOSE',
    values='MILES', aggfunc='mean').sort_index(ascending=False))
for start_col in range(0, len(task4.columns), 3):
    print(task4.iloc[:, start_col:start_col + 3].round(2).to_string())
    print()

PURPOSE   Customer Visit  Meal/Entertain  Meeting
CATEGORY                                         
Personal             NaN             NaN      NaN
Business            9.39            5.47     9.57

PURPOSE   Moving  Temporary Site  Не указана
CATEGORY                                    
Personal     6.1             NaN        6.87
Business     NaN            9.52       11.11



Интерпретация. Сводная таблица позволяет сравнить среднее расстояние между целями внутри деловых поездок. Для личных поездок заполнены только Moving и «Не указана». Среднее по Moving основано на одной поездке и не является устойчивой оценкой. Сопоставлять Business и Personal по остальным известным целям нельзя, поскольку соответствующих личных поездок в выборке нет.

## 7 Задание 5 Расстояние по категории длительности
Выделяются три категории: короткая — до 10 минут включительно; средняя — более 10 и до 30 минут включительно; длинная — более 30 минут. Это простые интерпретируемые границы для кратких, обычных и длительных поездок. Они выбраны для учебного анализа, не являются отраслевым стандартом и дают непустые группы.

Для каждой категории вычисляются среднее и медианное расстояние. Фраза «отфильтровать по убыванию среднего времени» трактуется как сортировка по среднему времени: фильтрация не задаёт порядок. Поэтому дополнительно вычисляется mean_duration_min, по которому и сортируется результат. Количество count помогает оценить размер групп.

In [10]:
labels = ['Короткая', 'Средняя', 'Длинная']
df['DURATION_CATEGORY'] = pd.cut(df['DURATION_MIN'],
    bins=[0, 10, 30, np.inf], labels=labels, right=True, include_lowest=True)
task5 = (df.groupby('DURATION_CATEGORY', observed=True)
    .agg(mean_miles=('MILES', 'mean'), median_miles=('MILES', 'median'),
         mean_duration_min=('DURATION_MIN', 'mean'), count=('MILES', 'size'))
    .sort_values('mean_duration_min', ascending=False))
print(task5.round(2).to_string())

                   mean_miles  median_miles  mean_duration_min  count
DURATION_CATEGORY                                                    
Длинная                 36.39          21.4              60.89     19
Средняя                  6.79           6.1              18.50     92
Короткая                 2.78           2.5               7.31     48


Интерпретация. При увеличении категории длительности растут среднее и медианное расстояние. В длинных поездках среднее заметно превышает медиану: на него влияют несколько больших расстояний, включая 195.3 мили. Поэтому медиана лучше описывает центральное значение этой группы. Сортировка выполнена именно по mean_duration_min, что видно из убывающего порядка значений.

## 8 Задание 6 Длительность по цели и категории расстояния
Категории расстояния: короткая — до 5 миль включительно; средняя — более 5 и до 15 миль включительно; длинная — более 15 миль. Границы отделяют небольшие местные перемещения от более протяжённых поездок, остаются понятными и дают непустые группы.

Создаётся сводная таблица: в строках цель поездки, в столбцах категория расстояния, в значениях средняя и медианная длительность в минутах. Для удобства чтения две статистики печатаются отдельными блоками одной сводной таблицы. Распределение по категориям зависит от гипотезы восстановления MILES.

In [11]:
df['MILES_CATEGORY'] = pd.cut(df['MILES'], bins=[0, 5, 15, np.inf],
    labels=labels, right=True, include_lowest=True)
task6 = df.pivot_table(index='PURPOSE', columns='MILES_CATEGORY',
    values='DURATION_MIN', aggfunc=['mean', 'median'], observed=True)
print('Средняя длительность, мин:')
print(task6['mean'].round(2).to_string())
print('\nМедианная длительность, мин:')
print(task6['median'].round(2).to_string())
print('\nКоличество поездок по категориям расстояния:')
print(df['MILES_CATEGORY'].value_counts(sort=False).to_string())

Средняя длительность, мин:
MILES_CATEGORY  Короткая  Средняя  Длинная
PURPOSE                                   
Customer Visit     11.54    19.31    45.25
Meal/Entertain      8.58    20.17    32.00
Meeting            17.00    21.40    51.00
Moving               NaN    21.00      NaN
Temporary Site     12.00    21.50    44.00
Не указана         10.18    21.47    79.12

Медианная длительность, мин:
MILES_CATEGORY  Короткая  Средняя  Длинная
PURPOSE                                   
Customer Visit      12.0     18.0     47.0
Meal/Entertain       8.0     19.0     28.0
Meeting             17.0     19.0     51.0
Moving               NaN     21.0      NaN
Temporary Site      12.0     21.5     44.0
Не указана           9.0     21.0     57.5

Количество поездок по категориям расстояния:
MILES_CATEGORY
Короткая    73
Средняя     68
Длинная     18


Интерпретация. Для целей, представленных в нескольких категориях расстояния, более длинные поездки в целом требуют больше времени. Среднее и медиана могут различаться из-за неоднородности длительности и отдельных больших значений. NaN обозначает отсутствующее сочетание признаков. Группы Moving и Temporary Site малочисленны, поэтому их статистики нельзя переносить на генеральную совокупность.

## 9 Собственное задание Группировка с фильтрацией
Задание: выбрать деловые поездки длительностью не менее 20 минут и с известной целью. По каждой цели рассчитать количество поездок, среднее расстояние, среднюю и медианную длительность; отсортировать по убыванию количества.

Порог 20 минут отделяет достаточно продолжительные поездки и близок к среднему времени всей выборки. Неизвестные цели исключаются только в этой частной задаче, так как сравниваются именно заявленные цели.

In [12]:
selected = df.loc[(df['CATEGORY'] == 'Business')
    & (df['DURATION_MIN'] >= 20) & (~df['PURPOSE_WAS_MISSING'])]
task7 = (selected.groupby('PURPOSE')
    .agg(count=('MILES', 'size'), mean_miles=('MILES', 'mean'),
         mean_duration_min=('DURATION_MIN', 'mean'),
         median_duration_min=('DURATION_MIN', 'median'))
    .sort_values(['count', 'mean_duration_min'], ascending=[False, False]))
print('Отобрано поездок:', len(selected))
print(task7.round(2).to_string())

Отобрано поездок: 26
                count  mean_miles  mean_duration_min  median_duration_min
PURPOSE                                                                  
Customer Visit     10       19.33              33.60                 27.5
Meal/Entertain      8       11.64              28.38                 28.5
Meeting             6       10.88              38.17                 28.5
Temporary Site      2       15.10              36.00                 36.0


Интерпретация. После фильтрации осталось 26 поездок. Наиболее многочисленная цель — Customer Visit (10 поездок). Результат относится только к деловым поездкам от 20 минут с известной целью; его нельзя напрямую сравнивать с частотами всей выборки. Средние расстояния сохраняют ограничение, связанное с восстановлением MILES.

## 10 Проверка и сохранение результатов
Проверяется полнота группировок, отсутствие пропусков в вычисленных категориях и соблюдение правил сортировки. Суммы количества должны совпадать с числом поездок соответствующей выборки. Округление до двух знаков используется при выводе; экспорт сохраняет исходную точность расчётов.

In [13]:
assert task1.sum() == len(df)
assert task2['count'].sum() == len(df)
assert task2['count'].is_monotonic_decreasing
assert task3['MILES'].is_monotonic_decreasing
assert task5['count'].sum() == len(df)
assert task5['mean_duration_min'].is_monotonic_decreasing
assert df[['DURATION_CATEGORY', 'MILES_CATEGORY']].notna().all().all()
assert task7['count'].sum() == len(selected)
df.to_csv('drivers_cleaned.csv', sep=';', encoding='utf-8-sig', index=False)
results = Path('results')
results.mkdir(exist_ok=True)
for i, table in enumerate([task1, task2, task3, task4, task5, task6, task7], 1):
    table.to_csv(results / f'task_{i}.csv', sep=';', encoding='utf-8-sig',
                 index=(i != 2))
print('Проверки пройдены. Сохранены drivers_cleaned.csv и 7 таблиц в results/.')

Проверки пройдены. Сохранены drivers_cleaned.csv и 7 таблиц в results/.


## 11 Вывод по работе
Освоены группировка groupby, агрегирование agg, построение сводных таблиц pivot_table, сортировка и категоризация cut. Выполнены все шесть заданий варианта 2 и собственная группировка с несколькими условиями фильтрации.

После удаления двух дублей анализируется 159 поездок. Из них 148, или 93,08%, относятся к Business; 11, или 6,92%, к Personal. Типичная длительность по медиане равна 15 минутам, средняя — 20,19 минуты. Различие показывает влияние длительных поездок на среднее. География выборки сильно сосредоточена на Morrisville.

При принятой гипотезе реконструкции среднее расстояние составляет 9,12 мили, медиана — 5,50 мили, максимум — 195,30 мили. Превышение среднего над медианой согласуется с наличием небольшого числа протяжённых поездок. Категоризация показывает рост среднего и медианного расстояния от коротких по времени поездок к длинным.

Ограничения существенны: у 77 поездок отсутствует цель; 133 расстояния восстановлены по гипотезе; одна поездка требует проверки согласованности времени и расстояния. Удаление неизвестных целей из всей выборки или замена повреждённых расстояний общим средним скрыли бы эти проблемы. Поэтому неизвестные цели сохранены отдельной группой, исходные расстояния и флаги оставлены в данных, а выводы о расстоянии сформулированы условно. Для окончательного подтверждения необходим исходный столбец MILES до преобразования в Excel.

## 12 Использованные материалы
1. Лабораторная работа 1 «Методы агрегации и группировки данных»: общие требования — страницы 1–2 и 12–13, вариант 2 — страницы 17–18, состав отчёта — страницы 101–102.
2. «Лекция.pdf»: введение в анализ данных и предварительная обработка.
3. Боженко В. В., Татарникова Т. М. Язык программирования Python для анализа данных. СПб.: ГУАП, 2023. Предварительный анализ, группировка и сводные таблицы.
4. Предоставленный файл drivers.csv. Внешние наборы данных в расчётах не использовались.

Отдельное условие контрольной работы, на которое ссылается методичка, не предоставлено. Здесь включена предобработка по доступным материалам; её соответствие отдельным пунктам контрольной следует проверить после получения того условия.